# Mundo dos Wumpus — Lógica Proposicional

Agente baseado em conhecimento que mantém uma base de conhecimento proposicional, infere células seguras a partir das percepções (brisa, fedor, brilho) e planeja a rota até o ouro.

**Técnica:** Representação fatorada e inferência lógica  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/10-mundo-dos-wumpus-logica-proposicional.ipynb)


In [ ]:
"""
=============================================================================
AGENTE MUNDO DOS WUMPUS COM LÓGICA PROPOSICIONAL
=============================================================================
Implementação de um Agente de Planejamento Baseado em Conhecimento (APBC)
que navega no Mundo dos Wumpus utilizando raciocínio lógico dedutivo.

Classificação do Agente:
  - Agente Reativo Simples: age apenas pela percepção atual (sem memória).
  - Agente Baseado em Modelo: mantém estado interno do mundo.
  - Agente de Planejamento Baseado em Conhecimento (ESTE): mantém uma Base
    de Conhecimento (KB), aplica regras lógicas, infere novos fatos e planeja
    ações com base em consequências futuras — não apenas reage.

Fundamentos Teóricos:
  - Representação fatorada: o estado do mundo é decomposto em variáveis
    proposicionais (P, W, G, B, S, Gl, V, Safe) por célula, em vez de um
    único "estado atômico" opaco. Isso permite compartilhamento de estrutura
    e inferência parcial sem observar todo o mundo.
  - Lógica proposicional: fórmulas com conectivos ∧, ∨, ¬, →.
  - Inferência dedutiva: a KB implica novos fatos por encadeamento de regras.
  - Satisfatibilidade (SAT) via DFS/backtracking: verificação de hipóteses.
=============================================================================
"""

import random
import copy
from typing import List, Tuple, Set, Dict, Optional, FrozenSet
from collections import deque
from enum import Enum, auto


# ---------------------------------------------------------------------------
# ENUMERAÇÕES E TIPOS
# ---------------------------------------------------------------------------

class Percepcao(Enum):
    BRISA   = auto()   # brisa → poço adjacente
    FEDOR   = auto()   # fedor → Wumpus adjacente
    BRILHO  = auto()   # brilho → ouro aqui
    IMPACTO = auto()   # colidiu com parede
    GRITO   = auto()   # Wumpus morto

class Acao(Enum):
    MOVER    = auto()
    GIRAR_ESQ = auto()
    GIRAR_DIR = auto()
    COLETAR  = auto()
    ATIRAR   = auto()
    SAIR     = auto()

class Direcao(Enum):
    NORTE = (0,  1)
    SUL   = (0, -1)
    LESTE = (1,  0)
    OESTE = (-1, 0)

    def delta(self) -> Tuple[int,int]:
        return self.value


# ---------------------------------------------------------------------------
# CLASSE: Celula
# ---------------------------------------------------------------------------
class Celula:
    """
    Representa uma célula da grade com suas variáveis proposicionais.

    Representação Fatorada:
      Em vez de um código atômico (ex: "estado_42"), cada célula expõe
      variáveis nomeadas. Isso permite que regras lógicas como
        B(x,y) → P(x+1,y) ∨ P(x-1,y) ∨ P(x,y+1) ∨ P(x,y-1)
      sejam aplicadas diretamente sem decodificação.

    Variáveis de estado por célula:
      has_pit    P(x,y)   — há poço
      has_wumpus W(x,y)   — há Wumpus
      has_gold   G(x,y)   — há ouro
      breeze     B(x,y)   — agente percebeu brisa aqui
      stench     S(x,y)   — agente percebeu fedor aqui
      glitter    Gl(x,y)  — agente percebeu brilho aqui
      visited    V(x,y)   — agente visitou
      safe       Safe(x,y)— inferido como seguro
    """
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y
        # Verdade do mundo (oculta ao agente)
        self.has_pit    = False
        self.has_wumpus = False
        self.has_gold   = False
        # Sinais gerados pelo mundo
        self.breeze  = False
        self.stench  = False
        self.glitter = False
        # Conhecimento do agente
        self.visited  = False
        self.safe     = False
        self.pit_possible    = True   # ainda pode ter poço (sem prova negativa)
        self.wumpus_possible = True   # ainda pode ter Wumpus

    def __repr__(self) -> str:
        return f"Celula({self.x},{self.y})"


# ---------------------------------------------------------------------------
# CLASSE: AmbienteWumpus
# ---------------------------------------------------------------------------
class AmbienteWumpus:
    """
    Implementa o Mundo dos Wumpus como uma grade N×N.

    Regras do ambiente:
      - Célula (0,0): entrada — sempre segura, sem poço nem Wumpus.
      - Poços: distribuídos aleatoriamente com probabilidade p por célula.
      - Um Wumpus: posicionado aleatoriamente (fora de (0,0)).
      - Um ouro: posicionado aleatoriamente.
      - Brisa gerada em todas células adjacentes a poços.
      - Fedor gerado em todas células adjacentes ao Wumpus.
    """

    def __init__(self, tamanho: int = 4, prob_poco: float = 0.2, seed: Optional[int] = None):
        self.N = tamanho
        self.prob_poco = prob_poco
        if seed is not None:
            random.seed(seed)
        self.grade: List[List[Celula]] = [
            [Celula(x, y) for y in range(tamanho)]
            for x in range(tamanho)
        ]
        self.wumpus_vivo = True
        self.ouro_coletado = False
        self._gerar_mundo()

    # ------------------------------------------------------------------
    def _gerar_mundo(self):
        """Posiciona poços, Wumpus e ouro; propaga brisa e fedor."""
        # Células candidatas (exceto origem)
        candidatas = [
            (x, y)
            for x in range(self.N)
            for y in range(self.N)
            if (x, y) != (0, 0)
        ]

        # Poços
        for x, y in candidatas:
            if random.random() < self.prob_poco:
                self.grade[x][y].has_pit = True

        # Wumpus — escolhe célula sem poço
        sem_poco = [(x,y) for x,y in candidatas if not self.grade[x][y].has_pit]
        if sem_poco:
            wx, wy = random.choice(sem_poco)
            self.grade[wx][wy].has_wumpus = True
            self.wumpus_pos = (wx, wy)
        else:
            # fallback: força Wumpus mesmo com poço
            wx, wy = random.choice(candidatas)
            self.grade[wx][wy].has_wumpus = True
            self.wumpus_pos = (wx, wy)

        # Ouro
        gx, gy = random.choice(candidatas)
        self.grade[gx][gy].has_gold = True
        self.gold_pos = (gx, gy)

        # Propaga brisa e fedor
        for x in range(self.N):
            for y in range(self.N):
                for ax, ay in self._adjacentes(x, y):
                    if self.grade[ax][ay].has_pit:
                        self.grade[x][y].breeze = True
                    if self.grade[ax][ay].has_wumpus:
                        self.grade[x][y].stench = True
                if self.grade[x][y].has_gold:
                    self.grade[x][y].glitter = True

        # Origem é segura por definição
        self.grade[0][0].safe = True

    # ------------------------------------------------------------------
    def _adjacentes(self, x: int, y: int) -> List[Tuple[int,int]]:
        """Retorna células adjacentes válidas (von Neumann)."""
        viz = []
        for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            nx, ny = x+dx, y+dy
            if 0 <= nx < self.N and 0 <= ny < self.N:
                viz.append((nx, ny))
        return viz

    # ------------------------------------------------------------------
    def perceber(self, x: int, y: int) -> Set[Percepcao]:
        """Retorna o conjunto de percepções na célula (x,y)."""
        c = self.grade[x][y]
        p: Set[Percepcao] = set()
        if c.breeze  and self.wumpus_vivo:
            pass  # brisa independe do Wumpus
        if c.breeze:
            p.add(Percepcao.BRISA)
        if c.stench and self.wumpus_vivo:
            p.add(Percepcao.FEDOR)
        if c.glitter and not self.ouro_coletado:
            p.add(Percepcao.BRILHO)
        return p

    # ------------------------------------------------------------------
    def mapa_verdadeiro(self) -> str:
        """Exibe o mapa real (apenas para debugging/comparação)."""
        linhas = ["\n  MAPA REAL (oculto ao agente):"]
        for y in range(self.N-1, -1, -1):
            linha = f"  {y} |"
            for x in range(self.N):
                c = self.grade[x][y]
                if c.has_wumpus:   linha += " W "
                elif c.has_pit:    linha += " P "
                elif c.has_gold:   linha += " G "
                else:              linha += " . "
            linhas.append(linha)
        linhas.append("     " + "".join(f" {x}  " for x in range(self.N)))
        return "\n".join(linhas)


# ---------------------------------------------------------------------------
# CLASSE: KnowledgeBase (Base de Conhecimento)
# ---------------------------------------------------------------------------
class KnowledgeBase:
    """
    Base de Conhecimento Proposicional do Agente.

    A KB é um conjunto de fórmulas proposicionais. Aqui representamos
    fórmulas como:
      - Fatos: literais positivos ou negativos (ex: "safe(1,2)", "~pit(2,3)")
      - Cláusulas disjuntivas: lista de literais (pelo menos um verdadeiro)
      - Inferências derivadas dinamicamente

    Regras Lógicas implementadas (expressas em lógica proposicional):

    [R1] B(x,y) → P(x+1,y) ∨ P(x-1,y) ∨ P(x,y+1) ∨ P(x,y-1)
         Se há brisa, ALGUM vizinho tem poço.

    [R2] ¬B(x,y) → ¬P(x+1,y) ∧ ¬P(x-1,y) ∧ ¬P(x,y+1) ∧ ¬P(x,y-1)
         Se NÃO há brisa, NENHUM vizinho tem poço.

    [R3] S(x,y) → W(x+1,y) ∨ W(x-1,y) ∨ W(x,y+1) ∨ W(x,y-1)
         Se há fedor, ALGUM vizinho tem Wumpus.

    [R4] ¬S(x,y) → ¬W(x+1,y) ∧ ¬W(x-1,y) ∧ ¬W(x,y+1) ∧ ¬W(x,y-1)
         Se NÃO há fedor, NENHUM vizinho tem Wumpus.

    [R5] Safe(x,y) ↔ ¬P(x,y) ∧ ¬W(x,y)
         Célula segura ↔ sem poço E sem Wumpus.
    """

    def __init__(self, tamanho: int):
        self.N = tamanho
        # Conjuntos de literais conhecidos (string "pit_x_y", "~pit_x_y", etc.)
        self.fatos: Set[str]     = set()
        # Cláusulas disjuntivas: frozenset de literais (pelo menos um verdadeiro)
        self.clausulas: List[FrozenSet[str]] = []
        # Células confirmadas sem poço e sem Wumpus
        self.sem_poco:    Set[Tuple[int,int]] = set()
        self.sem_wumpus:  Set[Tuple[int,int]] = set()
        self.safe_cells:  Set[Tuple[int,int]] = set()
        # Candidatas a poço/Wumpus (disjunções pendentes)
        self.possiveis_poco:   Dict[Tuple,bool] = {}  # (x,y) -> possível
        self.possiveis_wumpus: Dict[Tuple,bool] = {}
        # Registro de inferências para explicação
        self.log_inferencias: List[str] = []

        # (0,0) é seguro por axioma do ambiente
        self._marcar_seguro(0, 0, "axioma: célula de entrada sempre segura")

    # ------------------------------------------------------------------
    # Primitivos da KB
    # ------------------------------------------------------------------
    def afirmar(self, literal: str, justificativa: str = ""):
        """Adiciona um literal como fato à KB."""
        self.fatos.add(literal)
        neg = self._negar(literal)
        if neg in self.fatos:
            # Inconsistência (não deve ocorrer em ambiente correto)
            self.fatos.discard(neg)
        if justificativa:
            self.log_inferencias.append(f"  FATO: {literal}  [{justificativa}]")

    def afirmar_clausula(self, literais: List[str], justificativa: str = ""):
        """Adiciona cláusula disjuntiva (OR de literais)."""
        cl = frozenset(literais)
        if cl not in self.clausulas:
            self.clausulas.append(cl)
            if justificativa:
                self.log_inferencias.append(
                    f"  CLÁUSULA: ({' ∨ '.join(literais)})  [{justificativa}]"
                )

    def sabe(self, literal: str) -> bool:
        """Retorna True se o literal é um fato conhecido."""
        return literal in self.fatos

    @staticmethod
    def _negar(literal: str) -> str:
        return literal[1:] if literal.startswith("~") else "~" + literal

    @staticmethod
    def _lit_poco(x, y)    -> str: return f"pit_{x}_{y}"
    @staticmethod
    def _lit_wumpus(x, y)  -> str: return f"wumpus_{x}_{y}"
    @staticmethod
    def _lit_safe(x, y)    -> str: return f"safe_{x}_{y}"

    # ------------------------------------------------------------------
    def _adjacentes_validos(self, x: int, y: int) -> List[Tuple[int,int]]:
        viz = []
        for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            nx, ny = x+dx, y+dy
            if 0 <= nx < self.N and 0 <= ny < self.N:
                viz.append((nx, ny))
        return viz

    # ------------------------------------------------------------------
    # Atualização com percepções
    # ------------------------------------------------------------------
    def atualizar(self, x: int, y: int, percepcoes: Set[Percepcao]):
        """
        Aplica as regras lógicas R1-R4 para a célula (x,y) recém visitada.
        Atualiza a KB e executa inferências encadeadas.
        """
        self.log_inferencias.clear()
        adj = self._adjacentes_validos(x, y)

        # Marcar célula atual como visitada (portanto segura)
        self._marcar_seguro(x, y, f"agente visitou ({x},{y})")

        if Percepcao.BRISA in percepcoes:
            # R1: B(x,y) → P(adj1) ∨ P(adj2) ∨ ...
            lits = [self._lit_poco(ax, ay) for ax, ay in adj]
            if lits:
                self.afirmar_clausula(lits,
                    f"R1: brisa em ({x},{y}) → poço em algum de {adj}")
        else:
            # R2: ¬B(x,y) → ¬P(adj_i) para todo adj_i
            for ax, ay in adj:
                self.afirmar(f"~{self._lit_poco(ax,ay)}",
                    f"R2: sem brisa em ({x},{y}) → sem poço em ({ax},{ay})")
                self.sem_poco.add((ax, ay))

        if Percepcao.FEDOR in percepcoes:
            # R3: S(x,y) → W(adj1) ∨ W(adj2) ∨ ...
            lits = [self._lit_wumpus(ax, ay) for ax, ay in adj]
            if lits:
                self.afirmar_clausula(lits,
                    f"R3: fedor em ({x},{y}) → Wumpus em algum de {adj}")
        else:
            # R4: ¬S(x,y) → ¬W(adj_i) para todo adj_i
            for ax, ay in adj:
                self.afirmar(f"~{self._lit_wumpus(ax,ay)}",
                    f"R4: sem fedor em ({x},{y}) → sem Wumpus em ({ax},{ay})")
                self.sem_wumpus.add((ax, ay))

        # Inferência encadeada: simplificar cláusulas com novos fatos
        self._propagar()
        # Derivar novas células seguras
        self._derivar_seguros()

    # ------------------------------------------------------------------
    def _propagar(self):
        """
        Propagação de restrições: simplifica cláusulas disjuntivas
        usando fatos negativos conhecidos.

        Exemplo:
          Cláusula: {pit_1_2, pit_2_1}
          Novo fato: ~pit_2_1
          → Cláusula reduzida: {pit_1_2}  (unitária → fato!)

        Este é o núcleo do raciocínio dedutivo: encadeamento direto (forward chaining).
        """
        mudou = True
        while mudou:
            mudou = False
            novas_clausulas = []
            for cl in self.clausulas:
                # Remove literais falsificados por fatos negativos
                cl_reduzida = frozenset(
                    lit for lit in cl
                    if self._negar(lit) not in self.fatos
                )
                # Cláusula já satisfeita se algum literal é fato positivo
                if any(lit in self.fatos for lit in cl_reduzida):
                    continue  # descarta cláusula satisfeita
                if len(cl_reduzida) == 1:
                    # Cláusula unitária → literal deve ser verdadeiro
                    novo_lit = next(iter(cl_reduzida))
                    if novo_lit not in self.fatos:
                        self.afirmar(novo_lit, "propagação unitária")
                        mudou = True
                    # Não adiciona cláusula satisfeita
                elif len(cl_reduzida) == 0:
                    # Cláusula vazia → contradição (ambiente inconsistente)
                    pass
                else:
                    if cl_reduzida != cl:
                        mudou = True
                    novas_clausulas.append(cl_reduzida)
            self.clausulas = novas_clausulas

    # ------------------------------------------------------------------
    def _derivar_seguros(self):
        """
        Aplica R5: Safe(x,y) ↔ ¬P(x,y) ∧ ¬W(x,y)
        Marca como segura qualquer célula confirmada sem poço e sem Wumpus.
        """
        for x in range(self.N):
            for y in range(self.N):
                sem_p = (f"~{self._lit_poco(x,y)}" in self.fatos
                         or (x,y) in self.sem_poco)
                sem_w = (f"~{self._lit_wumpus(x,y)}" in self.fatos
                         or (x,y) in self.sem_wumpus)
                if sem_p and sem_w and (x,y) not in self.safe_cells:
                    self._marcar_seguro(x, y,
                        f"R5: ¬pit({x},{y}) ∧ ¬wumpus({x},{y}) → safe({x},{y})")

    # ------------------------------------------------------------------
    def _marcar_seguro(self, x: int, y: int, justificativa: str = ""):
        """Registra célula como segura na KB."""
        if (x,y) not in self.safe_cells:
            self.safe_cells.add((x,y))
            self.sem_poco.add((x,y))
            self.sem_wumpus.add((x,y))
            self.afirmar(f"~{self._lit_poco(x,y)}")
            self.afirmar(f"~{self._lit_wumpus(x,y)}")
            self.afirmar(self._lit_safe(x,y), justificativa)
            self.log_inferencias.append(f"  SEGURO: ({x},{y})  [{justificativa}]")

    # ------------------------------------------------------------------
    def eh_seguro(self, x: int, y: int) -> bool:
        return (x,y) in self.safe_cells

    def possivelmente_perigoso(self, x: int, y: int) -> bool:
        """
        Retorna True se a célula pode conter poço ou Wumpus
        (não foi provado seguro).
        """
        return (x,y) not in self.safe_cells

    # ------------------------------------------------------------------
    def localizar_wumpus(self) -> Optional[Tuple[int,int]]:
        """
        Tenta localizar o Wumpus por eliminação:
        Se apenas uma cláusula de fedor resta com um único candidato.
        """
        for cl in self.clausulas:
            lits_w = [l for l in cl if l.startswith("wumpus_")]
            if len(lits_w) == 1:
                partes = lits_w[0].split("_")
                return int(partes[1]), int(partes[2])
        return None


# ---------------------------------------------------------------------------
# MÓDULO: Satisfatibilidade via DFS/Backtracking
# ---------------------------------------------------------------------------
class SolverSAT:
    """
    Verificador de satisfatibilidade proposicional via DFS com backtracking.

    Dado um conjunto de cláusulas e fatos, verifica se uma hipótese H
    é consistente com a KB (KB ∧ H é satisfatível) ou se a KB implica H.

    Algoritmo DPLL simplificado:
      1. Propagação unitária: cláusulas com 1 literal → atribuição forçada.
      2. Eliminação de literais puros: literal que aparece só positivo/negativo.
      3. Escolha + backtracking: tenta True/False para variável não atribuída.

    Por que DFS?
      - SAT é NP-completo, mas espaços pequenos (grade 4×4 = 16 células,
        ~32 variáveis) são tratáveis.
      - DFS com backtracking explora sistematicamente atribuições.
      - É o motor clássico de inferência em lógica proposicional (IA-Aula5, Slide 73).
    """

    def __init__(self, clausulas: List[FrozenSet[str]], fatos: Set[str]):
        self.clausulas_base = clausulas
        self.fatos_base     = fatos

    def _simplificar(
        self,
        clausulas: List[FrozenSet[str]],
        atribuicao: Dict[str, bool]
    ) -> Optional[List[FrozenSet[str]]]:
        """
        Simplifica cláusulas com a atribuição atual.
        Retorna None se contradição, lista simplificada caso contrário.
        """
        resultado = []
        for cl in clausulas:
            nova_cl = set()
            satisfeita = False
            for lit in cl:
                var   = lit.lstrip("~")
                neg   = lit.startswith("~")
                valor = atribuicao.get(var)
                if valor is None:
                    nova_cl.add(lit)
                elif (valor and not neg) or (not valor and neg):
                    satisfeita = True
                    break
            if not satisfeita:
                if not nova_cl:
                    return None  # cláusula vazia → contradição
                resultado.append(frozenset(nova_cl))
        return resultado

    def _dpll(
        self,
        clausulas: List[FrozenSet[str]],
        atribuicao: Dict[str, bool]
    ) -> bool:
        """Recursão DPLL com propagação unitária e backtracking."""
        # Propagação unitária
        mudou = True
        while mudou:
            mudou = False
            for cl in clausulas:
                if len(cl) == 1:
                    lit = next(iter(cl))
                    var = lit.lstrip("~")
                    neg = lit.startswith("~")
                    val = not neg
                    if atribuicao.get(var) == (not val):
                        return False  # contradição
                    if var not in atribuicao:
                        atribuicao[var] = val
                        mudou = True
            clausulas = self._simplificar(clausulas, atribuicao)
            if clausulas is None:
                return False

        if not clausulas:
            return True  # todas satisfeitas

        # Escolha de variável livre
        var_livre = None
        for cl in clausulas:
            for lit in cl:
                var_livre = lit.lstrip("~")
                break
            if var_livre:
                break
        if var_livre is None:
            return True

        # Backtracking: tenta True e False
        for val in [True, False]:
            nova_atrib = dict(atribuicao)
            nova_atrib[var_livre] = val
            cl_simpl = self._simplificar(clausulas, nova_atrib)
            if cl_simpl is not None and self._dpll(cl_simpl, nova_atrib):
                return True
        return False

    def satisfativel(self, hipotese: str) -> bool:
        """
        Verifica se KB ∧ hipótese é satisfatível.
        Usado para testar se uma célula PODE ser perigosa.
        """
        # Converte fatos em cláusulas unitárias
        cls = list(self.clausulas_base) + [frozenset([h]) for h in self.fatos_base]
        cls.append(frozenset([hipotese]))
        atrib: Dict[str, bool] = {}
        # Aplica fatos como atribuições diretas
        for f in self.fatos_base:
            var = f.lstrip("~")
            atrib[var] = not f.startswith("~")
        return self._dpll(cls, atrib)

    def kb_implica(self, literal: str) -> bool:
        """
        Verifica se KB ⊨ literal usando refutação:
        KB ⊨ L  ↔  KB ∧ ¬L é insatisfatível.

        Este é o princípio da refutação: para provar L,
        assumimos ¬L e mostramos contradição.
        """
        neg = ("~" + literal) if not literal.startswith("~") else literal[1:]
        return not self.satisfativel(neg)


# ---------------------------------------------------------------------------
# CLASSE: Agente
# ---------------------------------------------------------------------------
class Agente:
    """
    Agente de Planejamento Baseado em Conhecimento.

    Diferenças em relação a outros tipos:
      - Reativo: if fedor → fugir. Sem memória, sem inferência.
      - Baseado em modelo: mantém estado interno, mas sem lógica formal.
      - ESTE (planejamento + conhecimento): usa KB formal, aplica regras
        lógicas, infere fatos novos, planeja sequência de ações futuras.

    Ciclo de raciocínio (TELL-ASK):
      1. TELL(KB, percepções) — atualiza KB com o que foi observado.
      2. ASK(KB, "qual célula é segura?") — consulta KB para decidir.
      3. Executa ação baseada na resposta.

    Estratégia de exploração:
      - Prioridade: células seguras não visitadas.
      - Fallback: células de risco calculado (menos candidatas a poço/Wumpus).
      - Sai quando coleta ouro ou esgota células seguras.
    """

    def __init__(self, ambiente: AmbienteWumpus):
        self.amb     = ambiente
        self.N       = ambiente.N
        self.kb      = KnowledgeBase(self.N)
        self.pos     = (0, 0)
        self.direcao = Direcao.LESTE
        self.vivo    = True
        self.tem_ouro = False
        self.tem_flecha = True
        self.passos   = 0
        self.pontuacao = 0
        self.historico: List[Tuple[int,int]] = [(0,0)]
        self.visitados: Set[Tuple[int,int]] = {(0,0)}
        self.MAX_PASSOS = self.N * self.N * 3

    # ------------------------------------------------------------------
    def _adjacentes(self, x: int, y: int) -> List[Tuple[int,int]]:
        viz = []
        for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            nx, ny = x+dx, y+dy
            if 0 <= nx < self.N and 0 <= ny < self.N:
                viz.append((nx, ny))
        return viz

    # ------------------------------------------------------------------
    def perceber_e_atualizar(self) -> Set[Percepcao]:
        """
        TELL: observa percepções e atualiza a KB.
        Implementa o ciclo de percepção → inferência → conhecimento.
        """
        x, y = self.pos
        percepcoes = self.amb.perceber(x, y)
        self.visitados.add((x,y))
        self.amb.grade[x][y].visited = True

        # Atualiza KB com percepções atuais
        self.kb.atualizar(x, y, percepcoes)

        # Verificação SAT adicional para células adjacentes
        solver = SolverSAT(self.kb.clausulas, self.kb.fatos)
        for ax, ay in self._adjacentes(x, y):
            lit_p = f"pit_{ax}_{ay}"
            lit_w = f"wumpus_{ax}_{ay}"
            # Se KB implica ~pit(ax,ay) E ~wumpus(ax,ay) → seguro por SAT
            if (solver.kb_implica(f"~{lit_p}") and
                solver.kb_implica(f"~{lit_w}")):
                self.kb._marcar_seguro(ax, ay, f"SAT: KB⊨¬pit∧¬wumpus em ({ax},{ay})")

        return percepcoes

    # ------------------------------------------------------------------
    def escolher_proxima_celula(self) -> Optional[Tuple[int,int]]:
        """
        ASK: consulta a KB para escolher a próxima célula a explorar.

        Prioridades (em ordem):
          1. Célula segura não visitada adjacente à posição atual.
          2. Qualquer célula segura não visitada (requer navegação).
          3. Retornar à origem (saída de segurança).
        """
        x, y = self.pos

        # Prioridade 1: seguras não visitadas adjacentes
        adj_seguros = [
            (ax, ay)
            for ax, ay in self._adjacentes(x, y)
            if self.kb.eh_seguro(ax, ay) and (ax,ay) not in self.visitados
        ]
        if adj_seguros:
            return adj_seguros[0]

        # Prioridade 2: qualquer seguro não visitado
        todos_seguros = [
            (sx, sy)
            for (sx, sy) in self.kb.safe_cells
            if (sx, sy) not in self.visitados
        ]
        if todos_seguros:
            # Escolhe o mais próximo por distância Manhattan
            return min(todos_seguros,
                       key=lambda c: abs(c[0]-x) + abs(c[1]-y))

        # Prioridade 3: retornar à origem
        return (0, 0) if (x, y) != (0, 0) else None

    # ------------------------------------------------------------------
    def _caminho_ate(self, destino: Tuple[int,int]) -> List[Tuple[int,int]]:
        """
        BFS para encontrar caminho entre posição atual e destino,
        usando apenas células seguras conhecidas.
        """
        origem = self.pos
        if origem == destino:
            return []
        fila = deque([[origem]])
        visitados_bfs: Set[Tuple[int,int]] = {origem}
        while fila:
            caminho = fila.popleft()
            ultimo  = caminho[-1]
            for nx, ny in self._adjacentes(*ultimo):
                if (nx,ny) == destino:
                    return caminho[1:] + [(nx,ny)]
                if (nx,ny) not in visitados_bfs and self.kb.eh_seguro(nx,ny):
                    visitados_bfs.add((nx,ny))
                    fila.append(caminho + [(nx,ny)])
        # Se não há caminho seguro, tenta qualquer caminho
        fila = deque([[origem]])
        visitados_bfs = {origem}
        while fila:
            caminho = fila.popleft()
            ultimo  = caminho[-1]
            for nx, ny in self._adjacentes(*ultimo):
                if (nx,ny) == destino:
                    return caminho[1:] + [(nx,ny)]
                if (nx,ny) not in visitados_bfs:
                    visitados_bfs.add((nx,ny))
                    fila.append(caminho + [(nx,ny)])
        return []

    # ------------------------------------------------------------------
    def mover(self, destino: Tuple[int,int]) -> bool:
        """
        Move o agente para a célula destino (passo a passo via BFS).
        Verifica perigos reais (morte por poço ou Wumpus).
        """
        caminho = self._caminho_ate(destino)
        if not caminho:
            return False

        for (nx, ny) in caminho:
            self.pos = (nx, ny)
            self.passos += 1
            self.pontuacao -= 1  # custo de movimento

            c = self.amb.grade[nx][ny]
            if c.has_pit:
                self.vivo = False
                self.pontuacao -= 1000
                return False
            if c.has_wumpus and self.amb.wumpus_vivo:
                self.vivo = False
                self.pontuacao -= 1000
                return False

        self.historico.append(self.pos)
        return True

    # ------------------------------------------------------------------
    def atirar_flecha(self) -> bool:
        """
        Opcional: atira flecha na direção do Wumpus inferido.
        Mata o Wumpus se a localização for correta.
        """
        if not self.tem_flecha:
            return False
        self.tem_flecha = False
        self.pontuacao -= 10

        alvo = self.kb.localizar_wumpus()
        if alvo and self.amb.grade[alvo[0]][alvo[1]].has_wumpus:
            self.amb.wumpus_vivo = False
            self.kb.sem_wumpus.add(alvo)
            self.kb.afirmar(f"~wumpus_{alvo[0]}_{alvo[1]}", "flecha acertou Wumpus")
            self.kb._derivar_seguros()
            return True
        return False

    # ------------------------------------------------------------------
    def executar(self) -> Dict:
        """
        Loop principal do agente: perceber → inferir → agir.

        Encadeamento lógico a cada passo:
          1. Percebe o ambiente (TELL).
          2. Infere células seguras (KB ⊢ inferências).
          3. Decide próxima ação (ASK).
          4. Executa ação e registra resultado.
        """
        resultados = {
            "ouro_coletado": False,
            "sobreviveu": True,
            "passos": 0,
            "pontuacao": 0,
            "log": []
        }

        _log = resultados["log"]

        def registrar(msg: str):
            _log.append(msg)

        registrar("=" * 62)
        registrar("  AGENTE MUNDO DOS WUMPUS — INÍCIO DA MISSÃO")
        registrar("=" * 62)

        while self.vivo and self.passos < self.MAX_PASSOS:
            x, y = self.pos
            registrar(f"\n{'─'*62}")
            registrar(f"  Passo {self.passos+1} | Posição: ({x},{y})")

            # 1. PERCEBER e TELL
            percepcoes = self.perceber_e_atualizar()
            p_str = ", ".join(p.name for p in percepcoes) or "Nenhuma"
            registrar(f"  Percepções: {p_str}")

            # 2. INFERÊNCIAS (log da KB)
            if self.kb.log_inferencias:
                registrar("  Inferências realizadas:")
                for inf in self.kb.log_inferencias:
                    registrar(inf)

            registrar(f"  Células seguras conhecidas: {sorted(self.kb.safe_cells)}")

            # 3. BRILHO → coletar ouro
            if Percepcao.BRILHO in percepcoes:
                self.tem_ouro = True
                self.amb.ouro_coletado = True
                self.pontuacao += 1000
                registrar("  ★ OURO COLETADO! Retornando à saída...")
                registrar(f"  Justificativa: Gl({x},{y}) percebido → ouro aqui.")
                # Retorna à origem
                if not self.mover((0,0)):
                    break
                registrar(f"  Agente retornou à saída ({self.pos}).")
                break

            # 4. Tentar atirar flecha se Wumpus localizado e tem flecha
            if self.tem_flecha and Percepcao.FEDOR in percepcoes:
                alvo = self.kb.localizar_wumpus()
                if alvo:
                    acertou = self.atirar_flecha()
                    registrar(f"  ► Flecha disparada em direção a {alvo}. "
                               f"{'Acertou!' if acertou else 'Errou.'}")

            # 5. DECIDIR próxima célula (ASK)
            proxima = self.escolher_proxima_celula()

            if proxima is None:
                registrar("  Nenhuma célula segura disponível. Retornando à saída.")
                self.mover((0,0))
                break

            registrar(f"  Decisão: mover para {proxima}")
            registrar(f"  Justificativa lógica: safe({proxima[0]},{proxima[1]}) ∈ KB "
                       f"{'(visitada)' if proxima in self.visitados else '(inferida)'}")

            # 6. MOVER
            if not self.mover(proxima):
                if not self.vivo:
                    registrar(f"  ✗ Agente morreu em {self.pos}!")
                    self.pontuacao -= 1000
                break

        resultados["ouro_coletado"] = self.tem_ouro
        resultados["sobreviveu"]    = self.vivo
        resultados["passos"]        = self.passos
        resultados["pontuacao"]     = self.pontuacao
        return resultados


# ---------------------------------------------------------------------------
# MÓDULO: Visualização
# ---------------------------------------------------------------------------
class Visualizador:
    """
    Exibe o estado do ambiente do ponto de vista do agente.

    Símbolos:
      ? → desconhecido
      S → seguro (inferido ou visitado)
      P → poço confirmado
      W → Wumpus confirmado
      G → ouro (se percebido brilho)
      A → posição atual do agente
      . → visitado e vazio
    """

    def __init__(self, agente: Agente):
        self.agente = agente

    def exibir(self, titulo: str = ""):
        N    = self.agente.N
        kb   = self.agente.kb
        pos  = self.agente.pos
        amb  = self.agente.amb

        if titulo:
            print(f"\n  {titulo}")
        print(f"\n  MAPA DO AGENTE ({N}×{N}):")
        print("  " + "┌" + "───┬"*(N-1) + "───┐")

        for y in range(N-1, -1, -1):
            linha = f"  {y} │"
            for x in range(N):
                if (x,y) == pos:
                    simbolo = " A "
                elif (x,y) in agente.visitados:
                    c = amb.grade[x][y]
                    if c.has_gold and not amb.ouro_coletado:
                        simbolo = " G "
                    else:
                        simbolo = " . "
                elif kb.eh_seguro(x,y):
                    simbolo = " S "
                elif kb.sabe(f"pit_{x}_{y}"):
                    simbolo = " P "
                elif kb.sabe(f"wumpus_{x}_{y}"):
                    simbolo = " W "
                else:
                    simbolo = " ? "
                linha += simbolo + "│"
            print(linha)
            if y > 0:
                print("  " + "├" + "───┼"*(N-1) + "───┤")
        print("  " + "└" + "───┴"*(N-1) + "───┘")
        print("  " + "".join(f"   {x}" for x in range(N)))
        print(f"  Legenda: A=agente  .=visitado  S=seguro  ?=desconhecido  P=poço  W=Wumpus  G=ouro")
        print(f"  Células seguras KB: {sorted(kb.safe_cells)}")
        print(f"  Pontuação atual: {self.agente.pontuacao}")


# ---------------------------------------------------------------------------
# SIMULADOR PRINCIPAL
# ---------------------------------------------------------------------------
def simular(
    tamanho: int   = 4,
    prob_poco: float = 0.2,
    seed: Optional[int] = None,
    verbose: bool  = True
) -> Dict:
    """
    Executa uma simulação completa do Mundo dos Wumpus.

    Parâmetros:
      tamanho   : lado da grade (ex: 4 → grade 4×4)
      prob_poco : probabilidade de poço por célula
      seed      : semente para reprodutibilidade
      verbose   : exibe log completo
    """
    print("\n")
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║   AGENTE MUNDO DOS WUMPUS — LÓGICA PROPOSICIONAL            ║")
    print("║   Agente de Planejamento Baseado em Conhecimento (APBC)      ║")
    print("╚══════════════════════════════════════════════════════════════╝")

    # Cria ambiente
    amb   = AmbienteWumpus(tamanho=tamanho, prob_poco=prob_poco, seed=seed)
    print(amb.mapa_verdadeiro())

    # Cria agente
    global agente
    agente = Agente(amb)
    viz    = Visualizador(agente)

    print(f"\n  Grade {tamanho}×{tamanho} | Wumpus em {amb.wumpus_pos} | Ouro em {amb.gold_pos}")
    print(f"  Agente inicia em (0,0) → conhecimento inicial: safe(0,0)")

    # Executa
    resultado = agente.executar()

    # Exibe log
    if verbose:
        print("\n" + "\n".join(resultado["log"]))

    # Exibe mapa final
    viz.exibir("ESTADO FINAL DO MAPA (visão do agente)")

    # Resultado
    print("\n" + "=" * 62)
    print("  RESULTADO DA MISSÃO")
    print("=" * 62)
    print(f"  Ouro coletado : {'SIM ★' if resultado['ouro_coletado'] else 'NÃO'}")
    print(f"  Sobreviveu    : {'SIM ✔' if resultado['sobreviveu'] else 'NÃO ✗'}")
    print(f"  Passos dados  : {resultado['passos']}")
    print(f"  Pontuação     : {resultado['pontuacao']}")
    print("=" * 62)

    return resultado


# ---------------------------------------------------------------------------
# DEMONSTRAÇÃO TEÓRICA: Regras Lógicas
# ---------------------------------------------------------------------------
def demonstrar_logica():
    """
    Demonstração pedagógica das regras lógicas e inferências.
    Mostra as fórmulas proposicionais e como são aplicadas.
    """
    print("\n" + "=" * 62)
    print("  DEMONSTRAÇÃO: REGRAS LÓGICAS PROPOSICIONAIS")
    print("=" * 62)

    regras = [
        ("R1 — Brisa implica poço adjacente",
         "B(x,y) → P(x+1,y) ∨ P(x-1,y) ∨ P(x,y+1) ∨ P(x,y-1)",
         "Se há brisa em (x,y), ALGUM vizinho tem poço (disjunção)."),
        ("R2 — Ausência de brisa exclui poços adjacentes",
         "¬B(x,y) → ¬P(x+1,y) ∧ ¬P(x-1,y) ∧ ¬P(x,y+1) ∧ ¬P(x,y-1)",
         "Se não há brisa, NENHUM vizinho tem poço (conjunção de negações)."),
        ("R3 — Fedor implica Wumpus adjacente",
         "S(x,y) → W(x+1,y) ∨ W(x-1,y) ∨ W(x,y+1) ∨ W(x,y-1)",
         "Se há fedor em (x,y), ALGUM vizinho tem Wumpus."),
        ("R4 — Ausência de fedor exclui Wumpus adjacente",
         "¬S(x,y) → ¬W(x+1,y) ∧ ¬W(x-1,y) ∧ ¬W(x,y+1) ∧ ¬W(x,y-1)",
         "Se não há fedor, NENHUM vizinho tem Wumpus."),
        ("R5 — Definição de célula segura",
         "Safe(x,y) ↔ ¬P(x,y) ∧ ¬W(x,y)",
         "Uma célula é segura sse não tem poço E não tem Wumpus."),
        ("R6 — Refutação para inferência",
         "KB ⊨ L  ↔  KB ∧ ¬L é insatisfatível (DPLL)",
         "Para provar L, assume ¬L e busca contradição via DFS."),
    ]

    for nome, formula, explicacao in regras:
        print(f"\n  [{nome}]")
        print(f"  Fórmula : {formula}")
        print(f"  Semântica: {explicacao}")

    print("\n  Exemplo de encadeamento (grade 4×4):")
    print("  Passo 1: Agente em (0,0). Sem brisa, sem fedor.")
    print("  KB recebe: ¬B(0,0), ¬S(0,0)")
    print("  R2 aplica: ¬P(1,0) ∧ ¬P(0,1)  →  (1,0) e (0,1) sem poço")
    print("  R4 aplica: ¬W(1,0) ∧ ¬W(0,1)  →  (1,0) e (0,1) sem Wumpus")
    print("  R5 aplica: Safe(1,0) ∧ Safe(0,1)  →  ambos seguros!")
    print("  Decisão: mover para (1,0) ou (0,1) — LOGICAMENTE SEGURO.")
    print("=" * 62)


# ---------------------------------------------------------------------------
# DEMONSTRAÇÃO: Solver SAT
# ---------------------------------------------------------------------------
def demonstrar_sat():
    """
    Demonstra o uso do solver SAT (DPLL) para verificar hipóteses.
    """
    print("\n" + "=" * 62)
    print("  DEMONSTRAÇÃO: SOLVER SAT (DPLL/DFS)")
    print("=" * 62)

    # Cenário: brisa em (0,1) mas não em (0,0). Candidatos: (1,1) ou (0,2)
    fatos: Set[str]     = {"~pit_0_0", "~pit_1_0", "~pit_0_1"}
    clausulas = [frozenset(["pit_1_1", "pit_0_2"])]   # R1 para (0,1)

    solver = SolverSAT(clausulas, fatos)

    print("  Cenário: brisa em (0,1). Poços confirmados fora: (0,0),(1,0),(0,1)")
    print("  Cláusula R1: pit(1,1) ∨ pit(0,2)")
    print()

    for celula, lit in [("(1,1)", "pit_1_1"), ("(0,2)", "pit_0_2")]:
        sat = solver.satisfativel(lit)
        impl_neg = solver.kb_implica(f"~{lit}")
        print(f"  pit{celula} satisfatível? {sat}  |  KB ⊨ ¬pit{celula}? {impl_neg}")
        status = "POSSIVELMENTE PERIGOSA" if sat else "SEGURA"
        print(f"  → Conclusão: {celula} é {status}")

    print("\n  Nota: Como ambas são satisfatíveis, o agente não pode provar")
    print("  que nenhuma delas é segura sem mais informação — correto!")
    print("=" * 62)


# ---------------------------------------------------------------------------
# EXPERIMENTO: Múltiplos mundos
# ---------------------------------------------------------------------------
def experimento_multiplos_mundos(n: int = 10):
    """Executa n simulações e computa taxa de sucesso."""
    print(f"\n  EXPERIMENTO: {n} MUNDOS ALEATÓRIOS")
    print("=" * 62)
    sucessos   = 0
    sobreviveu = 0
    pontuacoes = []

    for i in range(n):
        amb    = AmbienteWumpus(tamanho=4, prob_poco=0.2)
        ag     = Agente(amb)
        res    = ag.executar()
        if res["ouro_coletado"]:
            sucessos += 1
        if res["sobreviveu"]:
            sobreviveu += 1
        pontuacoes.append(res["pontuacao"])
        print(f"  Mundo {i+1:>2}: ouro={'SIM' if res['ouro_coletado'] else 'NÃO'} "
              f"| vivo={'SIM' if res['sobreviveu'] else 'NÃO'} "
              f"| pts={res['pontuacao']:>5} | passos={res['passos']}")

    media_pts = sum(pontuacoes) / len(pontuacoes)
    print(f"\n  Taxa de sucesso (ouro)    : {sucessos}/{n} ({100*sucessos/n:.0f}%)")
    print(f"  Taxa de sobrevivência      : {sobreviveu}/{n} ({100*sobreviveu/n:.0f}%)")
    print(f"  Pontuação média            : {media_pts:.1f}")
    print("=" * 62)


# ---------------------------------------------------------------------------
# PONTO DE ENTRADA
# ---------------------------------------------------------------------------
def main():
    print("\n")
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  MUNDO DOS WUMPUS — IA SIMBÓLICA E LÓGICA PROPOSICIONAL     ║")
    print("╚══════════════════════════════════════════════════════════════╝")

    # 1. Teoria: regras lógicas
    demonstrar_logica()

    # 2. Teoria: solver SAT
    demonstrar_sat()

    input("\n  [Enter para iniciar simulação principal...]")

    # 3. Simulação principal (seed fixada para reprodutibilidade)
    resultado = simular(tamanho=4, prob_poco=0.2, seed=42, verbose=True)

    # 4. Experimento com múltiplos mundos
    resp = input("\n  Executar experimento com múltiplos mundos? (s/n): ").strip().lower()
    if resp == "s":
        n = int(input("  Quantos mundos? (recomendado: 10-20): ").strip() or "10")
        experimento_multiplos_mundos(n)

    print("\n  Projeto concluído.\n")


if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════════╗
║  MUNDO DOS WUMPUS — IA SIMBÓLICA E LÓGICA PROPOSICIONAL     ║
╚══════════════════════════════════════════════════════════════╝

  DEMONSTRAÇÃO: REGRAS LÓGICAS PROPOSICIONAIS

  [R1 — Brisa implica poço adjacente]
  Fórmula : B(x,y) → P(x+1,y) ∨ P(x-1,y) ∨ P(x,y+1) ∨ P(x,y-1)
  Semântica: Se há brisa em (x,y), ALGUM vizinho tem poço (disjunção).

  [R2 — Ausência de brisa exclui poços adjacentes]
  Fórmula : ¬B(x,y) → ¬P(x+1,y) ∧ ¬P(x-1,y) ∧ ¬P(x,y+1) ∧ ¬P(x,y-1)
  Semântica: Se não há brisa, NENHUM vizinho tem poço (conjunção de negações).

  [R3 — Fedor implica Wumpus adjacente]
  Fórmula : S(x,y) → W(x+1,y) ∨ W(x-1,y) ∨ W(x,y+1) ∨ W(x,y-1)
  Semântica: Se há fedor em (x,y), ALGUM vizinho tem Wumpus.

  [R4 — Ausência de fedor exclui Wumpus adjacente]
  Fórmula : ¬S(x,y) → ¬W(x+1,y) ∧ ¬W(x-1,y) ∧ ¬W(x,y+1) ∧ ¬W(x,y-1)
  Semântica: Se não há fedor, NENHUM vizinho tem Wumpus.

  [R5 — Definição de célula segura]
  Fórm